# Trích xuất aspect và phân loại polarity — hardcore

Model scratch **chưa train**; dự đoán chỉ minh họa inference, không phải kết quả chất lượng. Bạn tự thiết kế tiền xử lý, thay model và viết training sau khi đọc luồng vào/ra này.

## 1. Load dữ liệu cơ bản

Tập inference: **test có sẵn của dataset**. Nguồn/revision trong SOURCE/DATASET_ID. Dữ liệu thô được giữ nguyên; thao tác nhóm/sắp ID chỉ để giữ giao thức mẫu của hai bản đầy đủ, không fit đặc trưng.

Chạy từ thư mục dự án, sửa DATA_DIR trước Run All. DATA_MODE="native" đọc nguồn đã khai báo; DATA_MODE="local_jsonl" đọc bản xuất **đúng nguồn**, schema: test.jsonl: text:str; có thể giữ các cột span/label/ordinal của nguồn, model chỉ nhận text. Đường dẫn ảnh trong JSONL phải hợp lệ từ thư mục chạy. Nếu dùng nguồn HF cần cài datasets; COCO cần torchvision/pycocotools; TFDS cần kernel TensorFlow tương thích. Không tự thay dataset khi thiếu dữ liệu.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import random
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from PIL import Image

SEED=42
BATCH_SIZE=16
MAX_INPUT_BYTES=4096  # Giới hạn đầu vào cố định của model; không đổi số mẫu inference.
MAX_OUTPUT_TOKENS=32
IMAGE_SIZE=64
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_MODE="native"  # Chọn local_jsonl rõ ràng để dùng bản xuất đúng dataset/schema bên dưới.

DATA_DIR=Path('data/absa_semeval2014_restaurants')
CACHE_DIR=Path("artifacts/absa_semeval2014_restaurants/hardcore/data_cache")
INFERENCE_SPLIT='test'
SOURCE={'kind': 'pasted_markdown_table', 'selected_rows': 1, 'task': 'Sequence Labeling / ABSA Extraction (Aspect Extraction)', 'metric': 'Aspect span F1 ↑; polarity F1 ↑', 'difficulty': 'Dễ'}
DATASET_ID='tomaarsen/setfit-absa-semeval-restaurants'
DATASET_REVISION='8885372fe73256f96bb60f65b550538ac5c26047'
LABELS=('negative', 'neutral', 'positive', 'conflict')


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Cung cấp JSONL từ đúng dataset/revision/schema đã ghi; không dùng dữ liệu giả thay thế.")
    with path.open(encoding="utf-8") as handle:
        rows=[json.loads(line) for line in handle if line.strip()]
    if not rows:raise ValueError(f"Tập dữ liệu rỗng: {path}")
    return rows

In [ ]:
if DATA_MODE=="local_jsonl":rows=read_jsonl(DATA_DIR/"test.jsonl")
elif DATA_MODE=="native":
    from datasets import load_dataset
    rows=load_dataset(DATASET_ID,revision=DATASET_REVISION,split="test",cache_dir=str(CACHE_DIR))
else:raise ValueError("DATA_MODE phải là native hoặc local_jsonl.")
# Nguồn có nhiều hàng aspect cho cùng câu; nhóm danh tính câu, không đọc gold span vào model.
texts=sorted({row["text"] for row in rows})
records=[{"id":hashlib.sha256(text.encode()).hexdigest(),"text":text} for text in texts]

assert records, "Không có mẫu test/validation để inference."
assert len({row["id"] for row in records})==len(records), "ID đầu vào bị trùng."
print("Tập inference:",INFERENCE_SPLIT,"; số mẫu:",len(records))

## 2. Định nghĩa một model và inference trực tiếp

Các lớp học đều khởi tạo ngẫu nhiên. Chỉ chuyển đầu vào tối thiểu bên trong model; không có tokenizer/vocabulary fit, hàm tiền xử lý riêng, loss, optimizer, training hay checkpoint. Giới hạn chiều đầu vào/đầu ra ở phần cấu hình là cố định và chỉ phục vụ model minh họa; toàn bộ mẫu của split vẫn được inference. Không truyền gold target vào forward.

In [ ]:
class RawAspectModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(256,16)
        self.tag=nn.Linear(16,3)
        self.polarity=nn.Linear(16,len(LABELS))
    def forward(self,texts):
        results=[]
        for text in texts:
            if not text:raise ValueError("Câu ABSA rỗng.")
            # Một bước cho mỗi ký tự, không đổi offset gốc và không cắt câu.
            ids=torch.tensor([ord(ch)%256 for ch in text],device=self.embedding.weight.device)
            hidden=self.embedding(ids)
            tags=self.tag(hidden).argmax(-1).tolist()
            spans=[]
            start=None
            for position,tag in enumerate(tags+[0]):
                if start is not None and tag!=2:
                    label=LABELS[int(self.polarity(hidden[start:position].mean(0)).argmax().item())]
                    spans.append({"start":start,"end":position,"text":text[start:position],"polarity":label})
                    start=None
                if position<len(tags) and (tag==1 or (tag==2 and start is None)):start=position
            results.append(spans)
        return results

In [ ]:
model=RawAspectModel().to(DEVICE)
model.eval()
predictions=[]
with torch.inference_mode():
    for start in range(0,len(records),BATCH_SIZE):
        batch=records[start:start+BATCH_SIZE]
        inputs=[row["text"] for row in batch]
        outputs=model(inputs)
        assert len(outputs)==len(batch), "Số dự đoán trong batch không khớp."
        predictions.extend([{'id':row["id"],'spans':output} for row,output in zip(batch,outputs)])

assert len(predictions)==len(records)>0
assert [row['id'] for row in predictions]==[row["id"] for row in records]
assert len({row['id'] for row in predictions})==len(records)
assert all(isinstance(row["spans"],list) and all(0<=s["start"]<s["end"]<=len(raw["text"]) and s["polarity"] in LABELS for s in row["spans"]) for raw,row in zip(records,predictions))

print(f"Đã inference {len(predictions)} mẫu trên {INFERENCE_SPLIT} bằng model scratch CHƯA TRAIN.")
print("Đầu ra minh họa thực thi, chưa đo chất lượng; cần tự xây dựng tiền xử lý, model và training.")
print(predictions[:2])
# File dự đoán chỉ được tạo khi người dùng chạy notebook; không cần checkpoint/helper.
PREDICTION_PATH=Path("artifacts/absa_semeval2014_restaurants/hardcore/predictions.json")
PREDICTION_PATH.parent.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")